# GPU 버전

In [1]:
import torch

torch.cuda.is_available()

True

In [2]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(torch.cuda.device_count())

NVIDIA GeForce RTX 3050 Laptop GPU
1


# 데이터셋 선택 및 하이퍼파라미터 설정
▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼

In [3]:
import time
import natsort
import os

folder_list = os.listdir("./data/")
# folder_list = ['bottle']
item_list = natsort.natsorted(folder_list)

print("다음 데이터셋들이 학습됩니다 : ", item_list)

다음 데이터셋들이 학습됩니다 :  ['cube']


In [4]:
#최소10, 200~400 추천, 10단위로 pth가 저장됨

epochs = 100
batch_size = 4

#3090 24GB에서 64까지 사용 가능했음
learning_rate = 0.005
image_size = 256

▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲

# Main.py

In [5]:
import torch
from dataset import get_data_transforms
from torchvision.datasets import ImageFolder
import numpy as np
import random
import os
from torch.utils.data import DataLoader
from resnet import resnet18, resnet34, resnet50, wide_resnet50_2
from de_resnet import de_resnet18, de_resnet34, de_wide_resnet50_2, de_resnet50
from dataset import RD_Dataset
import torch.backends.cudnn as cudnn
import argparse
from torch.nn import functional as F

In [6]:
def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [7]:
def loss_fucntion(a, b):
    #mse_loss = torch.nn.MSELoss()
    cos_loss = torch.nn.CosineSimilarity()
    loss = 0
    for item in range(len(a)):
        #print(a[item].shape)
        #print(b[item].shape)
        #loss += 0.1*mse_loss(a[item], b[item])
        loss += torch.mean(1-cos_loss(a[item].view(a[item].shape[0],-1),
                                      b[item].view(b[item].shape[0],-1)))
    return loss

In [8]:
def loss_concat(a, b):
    mse_loss = torch.nn.MSELoss()
    cos_loss = torch.nn.CosineSimilarity()
    loss = 0
    a_map = []
    b_map = []
    size = a[0].shape[-1]
    for item in range(len(a)):
        #loss += mse_loss(a[item], b[item])
        a_map.append(F.interpolate(a[item], size=size, mode='bilinear', align_corners=True))
        b_map.append(F.interpolate(b[item], size=size, mode='bilinear', align_corners=True))
    a_map = torch.cat(a_map,1)
    b_map = torch.cat(b_map,1)
    loss += torch.mean(1-cos_loss(a_map,b_map))
    return loss

In [9]:
def train(_class_):
    print(_class_)
        
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(device)

    data_transform = get_data_transforms(image_size, image_size)
    
    train_path = './data/' + _class_ + '/train'
    ckp_path = './checkpoints/' + 'wres50_'+_class_+'.pth'
    os.makedirs('./checkpoints', exist_ok=True)
    
    train_data = ImageFolder(root=train_path, transform=data_transform)
    train_dataloader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)

    encoder, bn = wide_resnet50_2(pretrained=True)
    encoder = encoder.to(device)
    bn = bn.to(device)
    encoder.eval()
    decoder = de_wide_resnet50_2(pretrained=False)
    decoder = decoder.to(device)

    optimizer = torch.optim.Adam(list(decoder.parameters())+list(bn.parameters()), lr=learning_rate, betas=(0.5,0.999))


    for epoch in range(epochs):
        start = time.time() 
        
        bn.train()
        decoder.train()
        loss_list = []
        for img, label in train_dataloader:
            img = img.to(device)
            inputs = encoder(img)
            outputs = decoder(bn(inputs))#bn(inputs))
            loss = loss_fucntion(inputs, outputs)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_list.append(loss.item())
        print('epoch [{}/{}], loss:{:.4f}'.format(epoch + 1, epochs, np.mean(loss_list)))
        print("time :",time.time() - start)  # 현재시각 - 시작시간 = 실행 시간

        # if (epoch + 1) % 2 == 0:
        # for cpu test
        if (epoch + 1) % 10 == 0:
            torch.save({'bn': bn.state_dict(),'decoder': decoder.state_dict()}, ckp_path)
            
    return loss

# 학습 시작

In [10]:
setup_seed(111)

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [11]:
#학습
for i in item_list:
    start_class = time.time()  # 시작 시간 저장

    train(i)
    print(i, "time :",time.time() - start_class)  # 현재시각 - 시작시간 = 실행 시간

cube
cuda
epoch [1/100], loss:0.7965
time : 14.043464660644531
epoch [2/100], loss:0.4571
time : 13.615041255950928
epoch [3/100], loss:0.3728
time : 13.66312861442566
epoch [4/100], loss:0.3235
time : 13.703076601028442
epoch [5/100], loss:0.2824
time : 13.795411586761475
epoch [6/100], loss:0.2599
time : 13.800514698028564
epoch [7/100], loss:0.2517
time : 13.814452886581421
epoch [8/100], loss:0.2219
time : 13.846490383148193
epoch [9/100], loss:0.2073
time : 13.859033823013306
epoch [10/100], loss:0.1921
time : 13.88454532623291
epoch [11/100], loss:0.1807
time : 13.881311893463135
epoch [12/100], loss:0.1753
time : 13.870260953903198
epoch [13/100], loss:0.1626
time : 13.883838415145874
epoch [14/100], loss:0.1641
time : 13.86821961402893
epoch [15/100], loss:0.1510
time : 13.880647659301758
epoch [16/100], loss:0.1450
time : 13.849718570709229
epoch [17/100], loss:0.1360
time : 13.844361066818237
epoch [18/100], loss:0.1366
time : 13.848075866699219
epoch [19/100], loss:0.1469
ti